# Evaluation

Wir laden die fertig berechneten Test-Predictions aus `06_prediction_2-model.ipynb` und analysieren wo das Modell gut und wo es schwächer ist.

**Benchmark:** Stop Mean Baseline MAE = 50.7s &nbsp;|&nbsp; **LightGBM v1 Test MAE = 46.3s** (−4.4s gegenüber Baseline ✅)

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import polars as pl
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("06_prediction_3-evaluation")


## Load Model & Predictions

In [ ]:
pred_path = Path(str(TEST)).parent / "test_predictions.parquet"
pred = pl.read_parquet(pred_path)

print(f"Predictions geladen: {len(pred):,} Zeilen")
print(f"Spalten: {pred.columns}")
pred.head(5)

## Metriken — Modell vs. Baseline

In [ ]:
def mae(df, actual='actual', predicted='predicted'):
    return (df[actual] - df[predicted]).abs().mean()

def rmse(df, actual='actual', predicted='predicted'):
    return ((df[actual] - df[predicted]) ** 2).mean() ** 0.5

def otp(df, actual='actual', predicted='predicted', threshold=60):
    return ((df[actual] - df[predicted]).abs() <= threshold).mean()

# Modell-Metriken auf Test-Set
model_mae  = mae(pred)
model_rmse = rmse(pred)
model_otp  = otp(pred)

# Baseline-Werte aus Baseline-Notebook (Stop Mean)
BASELINE_MAE  = 50.7
BASELINE_RMSE = 86.2  # aus Baseline-Notebook
BASELINE_OTP  = 0.731

results = pl.DataFrame({
    "":          ["Stop Mean Baseline", "LightGBM v1", "Gewinn"],
    "MAE (s)":   [BASELINE_MAE, round(model_mae, 1), round(BASELINE_MAE - model_mae, 1)],
    "RMSE (s)":  [BASELINE_RMSE, round(model_rmse, 1), round(BASELINE_RMSE - model_rmse, 1)],
    "OTP ±60s":  [f"{BASELINE_OTP:.1%}", f"{model_otp:.1%}", f"+{(model_otp - BASELINE_OTP):.1%}"],
})

show_df(results.to_pandas())

## Error Analysis

Wo liegt das Modell daneben? Wir schlüsseln den MAE auf nach:
- **Tageszeit** — Rush-Hour vs. Nacht
- **Linie** — welche Linien sind schwer vorherzusagen?
- **Wetter** — Schnee / Regen / normal
- **Monat** — saisonale Schwächen

In [ ]:
# --- MAE nach Stunde ---
mae_hour = (
    pred
    .with_columns((pl.col('actual') - pl.col('predicted')).abs().alias('abs_err'))
    .group_by('hour')
    .agg(pl.col('abs_err').mean().alias('MAE'), pl.len().alias('n'))
    .sort('hour')
)

fig = px.bar(mae_hour.to_pandas(), x='hour', y='MAE',
             title='MAE nach Tageszeit',
             labels={'hour': 'Stunde', 'MAE': 'MAE (s)'})
fig.add_hline(y=model_mae, line_dash='dash', line_color='gray',
              annotation_text=f'Gesamt MAE {model_mae:.1f}s')
fig.show()
show_df(mae_hour.sort('MAE', descending=True).to_pandas())

In [ ]:
# --- MAE nach Linie ---
mae_line = (
    pred
    .with_columns((pl.col('actual') - pl.col('predicted')).abs().alias('abs_err'))
    .group_by('line_name')
    .agg(pl.col('abs_err').mean().alias('MAE'), pl.len().alias('n'))
    .sort('MAE', descending=True)
)

fig = px.bar(mae_line.to_pandas(), x='line_name', y='MAE',
             title='MAE nach Linie',
             labels={'line_name': 'Linie', 'MAE': 'MAE (s)'})
fig.add_hline(y=model_mae, line_dash='dash', line_color='gray',
              annotation_text=f'Gesamt MAE {model_mae:.1f}s')
fig.show()
show_df(mae_line.to_pandas())

In [ ]:
# --- MAE nach Wetter ---
mae_weather = (
    pred
    .with_columns([
        (pl.col('actual') - pl.col('predicted')).abs().alias('abs_err'),
        pl.when(pl.col('has_snow')).then(pl.lit('Schnee'))
          .when(pl.col('has_rain')).then(pl.lit('Regen'))
          .otherwise(pl.lit('Normal')).alias('weather'),
    ])
    .group_by('weather')
    .agg(pl.col('abs_err').mean().alias('MAE'), pl.len().alias('n'))
    .sort('MAE', descending=True)
)

fig = px.bar(mae_weather.to_pandas(), x='weather', y='MAE',
             title='MAE nach Wetterbedingung',
             labels={'weather': 'Wetter', 'MAE': 'MAE (s)'})
fig.add_hline(y=model_mae, line_dash='dash', line_color='gray',
              annotation_text=f'Gesamt MAE {model_mae:.1f}s')
fig.show()
show_df(mae_weather.to_pandas())

In [ ]:
# --- MAE nach Monat ---
mae_month = (
    pred
    .with_columns((pl.col('actual') - pl.col('predicted')).abs().alias('abs_err'))
    .group_by('month')
    .agg(pl.col('abs_err').mean().alias('MAE'), pl.len().alias('n'))
    .sort('month')
)

fig = px.bar(mae_month.to_pandas(), x='month', y='MAE',
             title='MAE nach Monat (Test-Jahr 2025)',
             labels={'month': 'Monat', 'MAE': 'MAE (s)'})
fig.add_hline(y=model_mae, line_dash='dash', line_color='gray',
              annotation_text=f'Gesamt MAE {model_mae:.1f}s')
fig.show()
show_df(mae_month.to_pandas())

## Residuals — Systematischer Bias?

Schaut das Modell systematisch zu optimistisch (Vorhersage < Ist) oder zu pessimistisch (Vorhersage > Ist)?

**Mean Bias Error (MBE):** positiv = Modell überschätzt Delay · negativ = unterschätzt

In [ ]:
residuals = pred.with_columns(
    (pl.col('actual') - pl.col('predicted')).alias('residual')
)

mbe = residuals['residual'].mean()
print(f"Mean Bias Error (MBE): {mbe:.2f}s")
print(f"  > 0 = Modell unterschätzt Delay (zu optimistisch)")
print(f"  < 0 = Modell überschätzt Delay (zu pessimistisch)")

# Residual-Verteilung
sample = residuals.sample(n=min(50_000, len(residuals)), seed=42)
fig = px.histogram(sample.to_pandas(), x='residual', nbins=100,
                   title='Residual-Verteilung (actual − predicted)',
                   labels={'residual': 'Residual (s)', 'count': 'Anzahl'},
                   range_x=[-300, 300])
fig.add_vline(x=0, line_color='red', line_dash='dash')
fig.add_vline(x=mbe, line_color='orange',
              annotation_text=f'MBE {mbe:.1f}s')
fig.show()

## Konkrete Vorhersage — Das Szenario aus dem Overview

Live-Demonstration: Eine einzelne Eingabe → eine Vorhersage. Input exakt wie im Szenario aus `06_prediction_0-overview`.

In [ ]:
import lightgbm as lgb

model_path = Path(str(TEST)).parent.parent / "models" / "lgbm_v1.txt"
model = lgb.Booster(model_file=str(model_path))

# Szenario aus 06_prediction_0-overview:
# Dienstag 17:00 · Haltestelle Paradeplatz · Linie 11 · leichter Regen · kein Event
scenario = pd.DataFrame([{
    "line_name":          "11",
    "stop_name":          "Paradeplatz",
    "district_nr":        1,
    "temperature":        14.0,
    "precipitation":      1.5,
    "wind_speed":         12.0,
    "flood_intensity":    0,
    "event_type":         "none",
    "event_size":         0,
    "hour":               17,
    "weekday":            1,
    "month":              6,
    "year":               2025,
    "season":             "summer",
    "is_weekend":         False,
    "is_november":        False,
    "gtfs_year":          "j25",
    "has_rain":           True,
    "has_heavy_rain":     False,
    "has_snow":           False,
    "has_flood":          False,
    "is_hot":             False,
    "is_holiday":         False,
    "has_event":          False,
    "event_weight":       0,
    "dwell_time":         0,
    "n_lines_at_stop":    14,
    "n_stops_line":       30,
    "is_start_stop":      False,
    "is_end_stop":        False,
    "event_weight_x_hour": 0,
    "is_late_night_weekend": False,
}])

# Kategoriale Spalten setzen
for col in ['line_name', 'stop_name', 'event_type', 'season', 'gtfs_year']:
    scenario[col] = scenario[col].astype('category')

pred_val = model.predict(scenario)[0]
print(f"Szenario: Dienstag 17:00 · Paradeplatz · Linie 11 · leichter Regen")
print(f"Vorhergesagter Delay: {pred_val:.0f}s ({pred_val/60:.1f} min)")

## Fazit

In [ ]:
fazit = pl.DataFrame({
    "Modell":     ["Grand Mean", "Hour Mean", "Line Mean", "Stop Mean", "LightGBM v1"],
    "MAE (s)":    [50.7, 49.2, 48.1, 50.7, round(model_mae, 1)],
    "vs. Baseline": ["—", "—", "—", "Benchmark", f"-{BASELINE_MAE - model_mae:.1f}s ✅"],
})

show_df(fazit.to_pandas())

print()
print("Fazit:")
print(f"  LightGBM v1 erreicht MAE {model_mae:.1f}s auf dem Test-Set (2025).")
print(f"  Das Modell schlägt die Stop-Mean-Baseline ({BASELINE_MAE}s) um {BASELINE_MAE - model_mae:.1f}s.")
print(f"  Stärkste Schwäche: Rush-Hour und Schneetage (siehe Error Analysis).")